# Execution Modes

## References

- https://quantum.cloud.ibm.com/docs/en/guides/execution-modes
- https://quantum.cloud.ibm.com/docs/en/guides/choose-execution-mode
- https://quantum.cloud.ibm.com/docs/en/guides/run-jobs-batch
- https://quantum.cloud.ibm.com/docs/en/guides/run-jobs-session
- https://quantum.cloud.ibm.com/docs/en/guides/execution-modes-rest-api
- https://quantum.cloud.ibm.com/docs/en/guides/execution-modes-faq

## Job Mode

A single primitive request made without a context manager. Circuits and inputs are packaged as primitive unified blocs (PUBs) and submitted as an execution task on the quantum computer. To run in job mode, specify mode=backend when instantiating a primitive.

## Batch Mode

A multi-job manager for efficiently running experiments comprising multi-job workloads. These workloads are made up of independently executable jobs that have no conditional relationship with each other. With batch mode, users submit their jobs all at once.

**To run in batch mode, specify mode=batch when instantiating a primitive or run the job in a batch context manager**

The system parallelizes or threads the pre-processing (classical computing) step of each primitive job to more tightly package quantum execution across jobs, and then runs the quantum execution of each job in quick succession to deliver the most efficient results.

**How Batch Mode Works**

When you submit jobs in a batch, the system optimizes the workflow in two main stages:

- Parallel Pre-processing: The system threads or parallelizes the classical computing steps (the "prep work") for each job.

- Sequential Execution: Once pre-processed, the quantum executions are run in quick succession on the QPU to minimize idle time and deliver results more efficiently.

**Note**

- When batching, jobs are not guaranteed to run in the order they are submitted. Also, while your batch jobs will run as closely together as possible, they don't get exclusive access to the backend. Therefore, your batch jobs might run in parallel with other users' jobs if there is enough processing capacity on the QPU. Additionally, QPU calibration jobs could run between the batched jobs.

- The queuing time does not decrease for the first job submitted within a batch. Therefore, batches do not provide any benefits when running a single job.

## Session Mode

A dedicated window for running a multi-job workload. During this window, the user has exclusive access of the system and no other jobs can run - including calibration jobs. This allows users to experiment with variational algorithms in a more predictable way and even run multiple experiments simultaneously, taking advantage of parallelism in the stack. Using sessions helps avoid delays caused by queuing each job separately, which can be particularly useful for iterative tasks that require frequent communication between classical and quantum resources.

To run in session mode, specify mode=session when instantiating a primitive, or run the job in a session context manager. Allows learning a **noise model once** and reusing it across jobs (useful with PEA/PEC)

The queuing time does not decrease for the first job submitted within a session. Therefore, sessions do not provide any benefits when running a single job.

## Basic Workflow

The basic workflow for batches and sessions is similar:

1. The first job in a batch or session enters the normal queue. For batches, the entire batch of jobs is scheduled together.
2. When the first job starts running, the maximum time to live (TTL) timer starts, and does not stop or pause until the end is reached.
3. The interactive TTL timer starts after each job is completed. If there are no workload jobs ready within the interactive TTL window, the workload is temporarily deactivated and normal job selection resumes. A job can reactivate the deactivated workload if the batch or session has not reached its maximum TTL value. The job must go through the normal queue to reactivate the workload.
4. If the maximum TTL value is reached, the workload ends and any remaining queued jobs fail. Any job currently running won't run to completion if doing so would exceed the instance's cost limit.

### Wall clock time (Batch level)

This is measured by a standard clock on the wall. It starts the moment your batch moves from the queue to "Active" and stops when the batch closes.

- What it includes: The time the QPU is running, the time the classical computer is processing data, and even the tiny gaps of idle time between your jobs.

- The Limit (**max_time**): If you set this to 1 hour, the entire batch container self-destructs after 60 minutes of real-world time, regardless of how much work got done.

### Quantum time (Job level)

This is measured only when the microwave pulses are actually hitting the qubits. It is much more granular.

- What it includes: Only the Quantum Time (QPU dedication). It ignores the classical overhead and the time spent waiting for the next job in the batch to load.

- The Limit (**max_execution_time**): If you set this to 300 seconds, the job will be killed only if it occupies the actual quantum hardware for more than 5 minutes.

What happens if you set nothing?

- Batch: IBM assigns a Default Maximum TTL (wall-clock). If your jobs take longer than this default to run sequentially, the last few jobs will fail.

- Job: IBM uses a Service-calculated timeout (quantum time). It looks at your circuit's complexity and sets a "reasonable" limit (capped at 3 hours) so a buggy circuit doesn't loop forever. The service calculates an appropriate job timeout value based on the input circuits and options. This service-calculated timeout is capped at 3 hours to ensure fair device usage. If a max_execution_time is also specified for the job, the lesser of the two values is used.

Batches also have an interactive time to live (**interactive TTL**) value between jobs that cannot be configured. If you don't explicitly close a batch, it is deactivated after the interactive TTL expires and can be reactivated at any time until it reaches its **maximum TTL (max_time)**.

When a session is started, it is assigned a maximum TTL value that determines how long a session can run. After this TTL is reached, the session is terminated, any jobs that are already running continue running, and any queued jobs that remain in the session are put into a failed state. There is also an interactive TTL value that cannot be configured. If no session jobs are queued within that window, the session is temporarily deactivated.

## IBM Quantum — Choosing the Right Execution Mode

Utility-scale quantum workloads can take many hours. Execution modes let you balance **cost vs. time** by scheduling classical and quantum resources efficiently.

---

### Recommendations & Best Practices

- **Use Batch** by default for multiple independent jobs
- **Use Session** only for iterative algorithms or when exclusive QPU access is needed
- **Use Job** for any single primitive submission
- If each job uses **< 1 min of QPU time**, combine jobs into one larger job
- If each job uses **> 1 min of QPU time**, run multiple jobs in parallel (up to 5 classical jobs at once)
- Sessions can be optimized by breaking large jobs into smaller parallel ones

---

### Example Use Cases

#### Variational Algorithm (e.g., VQE)
> Prepare ansatz → Evaluate cost on QPU → Classical optimizer → Adjust params → Repeat

- Use **Session**: each iteration needs the previous result; re-queuing after every step is costly and risks device drift.

#### Comparing Error Mitigation Settings
> Build circuit → Submit jobs with different mitigation settings → Compare results

- Use **Batch**: all jobs are known upfront and are independent; batch runs them close together (good for fair comparison) at lower cost than a session.

In [2]:
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    Batch,
    SamplerV2 as Sampler,
    EstimatorV2 as Estimator,
)


service = QiskitRuntimeService()

In [3]:
backend = service.least_busy(operational=True, simulator=False)
batch = Batch(backend=backend)
estimator = Estimator(mode=batch)
sampler = Sampler(mode=batch)
# Close the batch because no context manager was used.
batch.close()

In [4]:
# OR - use context manager

from qiskit_ibm_runtime import (
    Batch,
    SamplerV2 as Sampler,
    EstimatorV2 as Estimator,
)

backend = service.least_busy(operational=True, simulator=False)
# maximum time to live is set using max_time. This timer starts when the batch starts. When the value is reached, the batch is closed. 
# Any jobs that are running will finish, but jobs still queued are failed.
with Batch(backend=backend, max_time="25m"):
    estimator = Estimator()
    sampler = Sampler()

- Default Maximum TTL for all paid plans instance type - 8 hours

- Default Maximum TTL for Open instance type - 10 minutes

A batch automatically closes when it exits the context manager. When the batch context manager is exited, the batch is put into "In progress, not accepting new jobs" status. This means that the batch finishes processing all running or queued jobs until the maximum TTL value is reached. After all jobs are completed, the batch is immediately closed. You cannot submit jobs to a closed batch.

In [6]:
with Batch(backend=backend) as batch:
    estimator = Estimator()
    sampler = Sampler()
    job1 = estimator.run([estimator_pub])
    job2 = sampler.run([sampler_pub])
    print(batch.details())
    
# The batch is no longer accepting jobs but the submitted job will run to completion.
result = job1.result()
result2 = job2.result()

In [7]:
# Manually closing the batch

batch = Batch(backend=backend)

# If using qiskit-ibm-runtime earlier than 0.24.0, change `mode=` to `batch=`
estimator = Estimator(mode=batch)
sampler = Sampler(mode=batch)
job1 = estimator.run([estimator_pub])
job2 = sampler.run([sampler_pub])
print(f"Result1: {job1.result()}")
print(f"Result2: {job2.result()}")

# Manually close the batch. Running and queued jobs will run to completion.
batch.close()

Session status can be one of the following:

- Pending: The session has not started or has been deactivated. The next session job needs to wait in the queue like other jobs.
- In progress, accepting new jobs: The session is active and accepting new jobs.
- In progress, not accepting new jobs: The session is active but not accepting new jobs. Job submission to the session is rejected, but outstanding session jobs will run to completion. The session is automatically closed once all jobs finish.
- Closed: The session's maximum timeout value has been reached or the session was explicitly closed.

- Session.close() means the session no longer accepts new jobs, but existing jobs run to completion.
- Session.cancel() cancels all pending session jobs.

If you are using the REST API directly:

- PATCH /sessions/{id} with accepting_jobs=False means the session no longer accepts new jobs, but existing jobs run to completion.
- DELETE /sessions/{id}/close cancels all pending session jobs.

## IBM Quantum — Execution Modes using REST API

You can run Qiskit primitive workloads via REST API in three execution modes: **Job**, **Session**, and **Batch**.  
The examples below use Python's `requests` module, but any REST-capable language/framework works.

---

### Job Mode

- A single **Estimator** or **Sampler** primitive request made without a context manager
- No session ID needed

---

### Session Mode

Sessions allow efficient multi-job iterative workloads by avoiding repeated queuing delays.  
> Not available for **Open Plan** users.

### Start a Session

```python
import json
import requests

sessionsUrl = "https://quantum.cloud.ibm.com/api/v1/sessions"
auth_id = "Bearer <YOUR_BEARER_TOKEN>"
backend = "<BACKEND_NAME>"
crn = "<SERVICE-CRN>"

headersList = {
  "Accept": "application/json",
  "Content-Type": "application/json",
  "Authorization": auth_id,
  "Service-CRN": crn
}

payload = json.dumps({
  "backend": backend,
  "mode": "dedicated",
})

response = requests.request("POST", sessionsUrl, data=payload, headers=headersList)
sessionId = response.json()['id']
print(response.json())
# Output: {'id': 'crw9s7cdbt40008jxesg'}
```

### Close a Session

> Always close a session when done — reduces wait time for other users.

```python
closureURL = "https://quantum.cloud.ibm.com/api/v1/sessions/" + sessionId + "/close"

headersList = {
  "Accept": "application/json",
  "Authorization": auth_id,
  "Service-CRN": crn
}

closure_response = requests.request("DELETE", closureURL, headers=headersList)
print("Session closure response ok?:", closure_response.ok, closure_response.text)
# Output: Session closure response ok?: True
```

---

### Batch Mode

Submit a batch by setting `"mode": "batch"` in the payload. All jobs are scheduled together — no repeated queuing.

```python
import json
import requests

sessionsUrl = "https://quantum.cloud.ibm.com/api/v1/sessions"

headersList = {
  "Accept": "application/json",
  "Authorization": auth_id,
  "Service-CRN": crn,
  "Content-Type": "application/json"
}

payload = json.dumps({
  "backend": backend,
  "instance": "hub1/group1/project1",
  "mode": "batch"
})

response = requests.request("POST", sessionsUrl, data=payload, headers=headersList)
sessionId = response.json()['id']
```

---

## Submitting Jobs in a Session

Once a session is active, pass `session_id` in each job payload to link it.  
> `<parameter values>` in a PUB can be a single value or a list; supports NumPy broadcasting.

### Estimator Examples

| Config | PUBs structure |
|--------|---------------|
| 1 circuit, 4 observables | `[[circuit, [obs1, obs2, obs3, obs4]]]` |
| 1 circuit, 4 observables, 2 param sets | `[[circuit, [[obs1],[obs2],[obs3],[obs4]], [[vals1],[vals2]]]]` |
| 2 circuits, 2 observables | `[[circuit, obs1], [circuit, obs2]]` |

```python
job_input = {
  'program_id': 'estimator',
  "backend": backend,
  "session_id": sessionId,
  "params": {
    "pubs": [[resulting_qasm, [obs1, obs2, obs3, obs4]]],
    "options": {
      "transpilation": {"optimization_level": 1},
      "twirling": {"enable_gates": True, "enable_measure": True},
      # "dynamical_decoupling": {"enable": True, "sequence_type": "XpXm"},  # optional
    },
  }
}
```

### Sampler Examples

| Config | PUBs structure |
|--------|---------------|
| 1 circuit, no params | `[[circuit]]` |
| 1 circuit, 3 param sets | `[[circuit, [vals1, vals2, vals3]]]` |
| 2 circuits, 1 param set | `[[circuit, [val1]], [circuit, None, 100]]` |

```python
job_input = {
  'program_id': 'sampler',
  "backend": backend,
  "session_id": sessionId,
  "params": {
    "pubs": [[resulting_qasm]],
    "options": {
      "transpilation": {"optimization_level": 1},
      "twirling": {"enable_gates": True, "enable_measure": True},
      # "dynamical_decoupling": {"enable": True, "sequence_type": "XpXm"},  # optional
    },
  }
}
```